In [1]:


import pandas as pd
import numpy as np


In [2]:
data=pd.read_csv("powerplant_data.csv")

In [3]:
data.head()

,AT,V,AP,RH,PE
0,8.34,40.77,1010.84,90.01,480.48
1,23.64,58.49,1011.40,74.20,445.75
2,29.74,56.90,1007.15,41.91,438.76
3,19.07,49.69,1007.22,76.79,453.09
4,11.80,40.66,1017.13,97.20,464.43


In [4]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 9568 entries, 0 to 9567
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AT      9568 non-null   float64
 1   V       9568 non-null   float64
 2   AP      9568 non-null   float64
 3   RH      9568 non-null   float64
 4   PE      9568 non-null   float64
dtypes: float64(5)
memory usage: 373.9 KB


In [5]:
data.isna().sum()

AT    0
V     0
AP    0
RH    0
PE    0
dtype: int64

In [6]:
data.describe()

,AT,V,AP,RH,PE
count,9568.000000,9568.000000,9568.000000,9568.000000,9568.000000
mean,19.651231,54.305804,1013.259078,73.308978,454.365009
std,7.452473,12.707893,5.938784,14.600269,17.066995
min,1.810000,25.360000,992.890000,25.560000,420.260000
25%,13.510000,41.740000,1009.100000,63.327500,439.750000
50%,20.345000,52.080000,1012.940000,74.975000,451.550000
75%,25.720000,66.540000,1017.260000,84.830000,468.430000
max,37.110000,81.560000,1033.300000,100.160000,495.760000


In [7]:
x=data.drop("PE",axis=1)
y=data["PE"]

In [8]:
#split our data

from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.3,random_state=42)

In [9]:
from sklearn.preprocessing import StandardScaler

In [29]:
scaler=StandardScaler()
x_train_scaler=scaler.fit_transform(x_train)
x_test_scaler=scaler.transform(x_test)

In [30]:
x_train_scaler

array([[-1.71269637, -1.04514402,  1.49558946,  0.96375075],
       [-0.80906663, -0.8909988 , -0.96965459,  1.05438161],
       [ 0.15346638,  0.37126182, -0.1259708 ,  0.80995293],
       ...,
       [-0.21601778, -0.83516048,  0.36167844, -0.84543342],
       [ 0.95267669,  1.14198793, -0.42294749, -0.46437186],
       [-1.76892222, -1.19063824,  1.91405662,  0.90882296]],
      shape=(6697, 4))

In [31]:
import torch
import torch.nn as nn

In [32]:
x_train_tensor=torch.tensor(x_train_scaler,dtype=torch.float32)
y_train_tensor=torch.tensor(y_train.values,dtype=torch.float32).view(-1,1)

x_test_tensor=torch.tensor(x_test_scaler,dtype=torch.float32)
y_test_tensor=torch.tensor(y_test.values,dtype=torch.float32).view(-1,1)



In [33]:
from torch.utils.data import TensorDataset,DataLoader
train_dataset=TensorDataset(x_train_tensor,y_train_tensor)
test_dataset=TensorDataset(x_test_tensor,y_test_tensor)

In [36]:
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=32)

In [41]:
# build ANN
class ANN(nn.Module):
    def __init__(self):
        super(ANN,self).__init__()
        self.model=nn.Sequential(
        # 1st layer
            nn.Linear(x_train.shape[1],6),
            nn.ReLU(),

            nn.Linear(6,6),
            nn.ReLU(),

            nn.Linear(6,1),  
        
        )

    def forward(self,x):
        return self.model(x)
        

In [44]:
import torch.optim as optim
model=ANN()
# loss , optimizer
crietrion=nn.MSELoss()
optimizer=optim.Adam(model.parameters())

In [54]:
# Train the ANN
train_losses = []
val_losses = []

best_val_loss = float("inf")

epochs = 100

for epoch in range(epochs):
    model.train()
    running_loss = 0.0 # tot training loss for 1 epoch
    
    for xb, yb in train_loader:
        # xb = features of 1 batch
        # yb = labels of 1 batch
        optimizer.zero_grad()
        
        outputs = model(xb) # forward prop....predicted outputs for this batch
        loss = crietrion(outputs, yb) # compute loss
        loss.backward() # back prop.. compute gradients
        optimizer.step() # params update
        
        running_loss += loss.item() # loss is a tensor => py float

    epoch_train_loss = running_loss / len(train_loader)
    train_losses.append(epoch_train_loss)


    # Validation
    model.eval()
    running_val_loss = 0.0

    with torch.no_grad(): # no gradients compute
        for xb, yb in test_loader:
            outputs = model(xb)
            loss = crietrion(outputs, yb)
            running_val_loss += loss

    epoch_val_loss = running_val_loss / len(test_loader)
    val_losses.append(epoch_val_loss)

    print(f"epoch {epoch+1}/{epochs} ==> train loss = {epoch_train_loss} & val loss = {epoch_val_loss}")

    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), "best_model.pt") #.pt or .pth

epoch 1/100 ==> train loss = 772.1067862374442 & val loss = 754.712646484375
epoch 2/100 ==> train loss = 770.7480430966332 & val loss = 748.83447265625
epoch 3/100 ==> train loss = 762.5612354096912 & val loss = 741.5755004882812
epoch 4/100 ==> train loss = 754.1482529413132 & val loss = 732.0489501953125
epoch 5/100 ==> train loss = 745.9975728352864 & val loss = 724.0869140625
epoch 6/100 ==> train loss = 736.1856206984747 & val loss = 714.3355102539062
epoch 7/100 ==> train loss = 727.2978800455729 & val loss = 705.1324462890625
epoch 8/100 ==> train loss = 718.0355478922526 & val loss = 693.6168212890625
epoch 9/100 ==> train loss = 707.2470575241815 & val loss = 680.292724609375
epoch 10/100 ==> train loss = 691.6961616152809 & val loss = 667.2587890625
epoch 11/100 ==> train loss = 678.557694498698 & val loss = 652.293701171875
epoch 12/100 ==> train loss = 662.4197458902995 & val loss = 637.1104736328125
epoch 13/100 ==> train loss = 646.29185660226 & val loss = 620.8487548828

In [49]:
# !pip install matplotlib
# import matplotlib.pyplot as plt

# loss_df = pd.DataFrame({
#     "Training Loss": train_losses,
#     "Validation Loss": val_losses
# })

# plt.plot(loss_df["Training Loss"], label = "Training Loss")
# plt.plot(loss_df["Validation Loss"], label = "Validation Loss")

# plt.xlabel("Epochs")
# plt.ylabel("Losses")

# plt.legend()

In [50]:
model.load_state_dict(torch.load("best_model.pt"))

<All keys matched successfully>

In [55]:
# Evaluation

model.eval()
with torch.no_grad():
    train_preds = model(x_train_tensor)
    test_preds = model(x_test_tensor)

    train_mse_loss = crietrion(train_preds, y_train_tensor)
    test_mse_loss = crietrion(test_preds, y_test_tensor)

print("Training MSE:", train_mse_loss.item())
print("Testing MSE:", test_mse_loss.item())

Training MSE: 21.139684677124023
Testing MSE: 20.0948543548584


In [56]:
from sklearn.metrics import r2_score

print("r^2 score =", r2_score(y_test, test_preds))

r^2 score = 0.9302182058442701
